> **⚙️ Setup cell added by [doQumentation](https://doqumentation.org)**
>
> The code cell below was added automatically to install required packages
> (skipped if already installed, e.g. on Binder/Code Engine).
> It also contains a commented-out template for IBM Quantum credentials.
> [Learn more about automatic modifications.](https://doqumentation.org/about/code-modifications)

In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Tensor-network error mitigation (TEM): A Qiskit Function by Algorithmiq

*See the [API reference](/docs/api/functions/algorithmiq-tem)*

> **Note:** Qiskit Functions are an experimental feature available only to IBM Quantum&reg; Premium Plan, Flex Plan, and On-Prem (via IBM Quantum Platform API) Plan users. They are in preview release status and subject to change.

<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.5.1
```
</AccordionItem>
</Accordion>

## Overview

Algorithmiq’s Tensor-network Error Mitigation (TEM) method is a hybrid
quantum-classical algorithm designed for performing noise mitigation entirely at
the classical post-processing stage. With TEM, the user can compute the
expectation values of observables mitigating the inevitable noise-induced errors
that occur on quantum hardware with increased accuracy and cost efficiency,
making it a highly attractive option for quantum researchers and industry
practitioners alike.

The method consists of constructing a tensor network representing the inverse of
the global noise channel affecting the state of the quantum processor and then
applying the map to informationally complete measurement outcomes acquired from
the noisy state to obtain unbiased estimators for the observables.

As an advantage, TEM leverages informationally complete measurements to give
access to a vast set of mitigated expectation values of observables and has
optimal sampling overhead on the quantum hardware, as described in Filippov et
al. (2023), [arXiv:2307.11740](https://arxiv.org/abs/2307.11740), and  Filippov
et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542). The
measurement overhead refers to the number of additional measurements required to
perform efficient error mitigation, a critical factor in the feasibility of
quantum computations. Therefore, TEM has the potential to enable quantum
advantage in complex scenarios, such as applications in the fields of quantum
chaos, many-body physics, Hubbard dynamics, and small molecule chemistry
simulations.

The main features and benefits of TEM can be summarized as:

1.  **Optimal measurement overhead**: TEM is optimal with respect to
[theoretical bounds](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601),
meaning that no method can achieve a smaller measurement overhead. In other
words, TEM requires the minimum number of additional measurements to perform
error mitigation. This in turns means that TEM uses minimal quantum runtime.
2.  **Cost-effectiveness**: Since TEM handles noise mitigation entirely in the
post-processing stage, there is no need to add extra circuits to the quantum
computer, which not only makes the computation cheaper but also diminishes the
risk of introducing additional errors due to the imperfections of quantum
devices.
3.  **Estimation of multiple observables**: Thanks to informationally-complete
measurements, TEM efficiently estimates multiple observables with the same
measurement data from the quantum computer.
4.  **Measurement error mitigation**: The TEM Qiskit Function also includes a
[proprietary measurement error mitigation method](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154)
able to significantly reduce readout errors after a short calibration run.
5.  **Accuracy**: TEM significantly improves the accuracy and reliability of
digital quantum simulations, making quantum algorithms more precise and
dependable.

## Description

The TEM function allows you to obtain error-mitigated expectation values for
multiple observables on a quantum circuit with minimal sampling overhead. The
circuit is measured with an informationally complete positive operator-valued
measure (IC-POVM), and the collected measurement outcomes are processed on a
classical computer. This measurement is used to perform the tensor network
methods and build a noise-inversion map. The function applies a map that fully
inverts the whole noisy circuit using tensor networks to represent the noisy
layers.

![TEM schematics](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "Error-mitigated estimation of an observable O via post-processing measurement outcomes of the noisy quantum processor. U and N denote an ideal quantum operation and the associated noise map, which can be generally non-local (and extended to grey boxes). D stands for a tensor of operators that are dual to the effects in the IC measurement. The noise mitigation module M is a tensor network that is efficiently contracted from the middle out. The first iteration of the contraction is represented by the dotted purple line, the second one by the dashed line, and the third one by the solid line.")

Once the circuits are submitted to the function, they are transpiled and
optimized to minimize the number of layers with two-qubit gates (the noisier
gates on quantum devices). The noise affecting the layers is learned through
[Qiskit Runtime](/docs/api/qiskit-ibm-runtime/noise-learner-noise-learner)
using a sparse Pauli-Lindblad noise model as described in E. van den Berg, Z.
Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

The noise model is an accurate description of the noise on the device able to
capture subtle features, including qubit cross-talk. However, noise on the
devices can fluctuate and drift and the learned noise might not be accurate at
the point at which the estimation is done. This might result in inaccurate
results.

## Get started

Authenticate using your [IBM Quantum Platform API key](http://quantum.cloud.ibm.com/), and select the TEM function as follows. (This snippet assumes you've already [saved your account](/docs/guides/functions-get-started#install-qiskit-functions-catalog-client) to your local environment.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")
# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# Load your function
tem = catalog.load(tem_function_name)

## Example


The following snippet shows an example where TEM is used to compute the expectation values of an observable given a simple quantum circuit.

In [3]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device
# reported by IBM if not specified
backend_name = "ibm_marrakesh"

# Run the TEM function (uses around three minutes of QPU time)
job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Use the Qiskit Serverless APIs to check your Qiskit Function workload's status:

In [4]:
# Print the ID so you can use it later, if necessary
print(job.job_id)
print(job.status())

ea1d7802-b067-4cc9-af39-3c2402d22413


QUEUED


You can return the results as:

In [5]:
result = job.result()
evs = result[0].data.evs
print(evs[0])

-0.10628832892063486


> **Info:** The expected value for the noiseless circuit for the given operator should be around `0.18409094298943401`.

## Get support

Reach out to [qiskit_ibm@algorithmiq.fi](mailto:qiskit_ibm@algorithmiq.fi)

Be sure to include the following information:
- Qiskit Function Job ID (`qiskit-ibm-catalog`), `job.job_id`
- A detailed description of the issue
- Any relevant error messages or codes
- Steps to reproduce the issue

## Next steps

> **Tip:** - [Request access to Algorithmiq Tensor-network error mitigation](https://quantum.ibm.com/functions?id=4b1b9d76-c18b-4788-b70b-15125111fbe6).
> - Visit the [API reference](/docs/api/functions/algorithmiq-tem) for this Qiskit Function.